<a href="https://colab.research.google.com/github/RoseWang-web/Data_Portfolio/blob/main/notebooks/TaiwanPandemicPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("burgerwu/taiwan-covid19-dataset")

print("Path to dataset files:", path)

100%|██████████| 4.22M/4.22M [00:00<00:00, 51.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/burgerwu/taiwan-covid19-dataset/versions/5


In [9]:
import pandas as pd
import os

# Define the base directory where step 1 downloaded your files
base_path = "/root/.cache/kagglehub/datasets/burgerwu/taiwan-covid19-dataset/versions/5"

# Let's load the main time-series cases file
file_name = "covid19_tw_cases_2022Oct.csv"
full_path = os.path.join(base_path, file_name)

# Read it into a pandas DataFrame
df = pd.read_csv(full_path)

# Let's see the first 5 records!
print("First 5 records:")
print(df.head())

First 5 records:
  Date_Confirmation County_Living  Gender  Imported Age_Group  \
0        2020-01-22      Imported  female         1     55-59   
1        2020-01-24      Imported  female         1     50-54   
2        2020-01-24      Imported    male         1     55-59   
3        2020-01-26      Imported  female         1     55-59   
4        2020-01-27      Imported  female         1     50-54   

   Number_of_Confirmed_Cases  
0                          1  
1                          1  
2                          1  
3                          1  
4                          1  


In [25]:
import pandas as pd

# Since you uploaded it directly to Colab, the file path is just the name!
file_path = "figshare_taiwan_covid.xlsx"

# 1. Load the individual patient tracking data (parsing columns with dates)
individual_date_cols = [
    'date_of_departure_1', 'date_of_arrival_1', 'date_of_departure_2',
    'date_of_arrival_2', 'date_of_departure_3', 'date_of_arrival_3',
    'date_of_departure_4', 'date_of_arrival_4', 'date_of_transit',
    'date_of_arrival_to_taiwan', 'onset_of_symptom', 'report_to_cdc',
    'confirmed_date', 'icu', 'recovery', 'death_date',
    'date_of_contact_with_infected_case', 'earliest_infection_date',
    'negative_test_date_1', 'negative_test_date_2', 'negative_test_date_3', 'negative_test_date_4'
]
df_individual = pd.read_excel(file_path, sheet_name='Individual Data', parse_dates=individual_date_cols)

# 2. Load the daily summary metrics
df_summary = pd.read_excel(file_path, sheet_name='Summary', parse_dates=['announce_date'])

# Verify the datasets loaded successfully
print("=== Datasets Loaded Successfully ===")
print(f"Individual Data Rows: {df_individual.shape[0]}")
print(f"Summary Data Rows:    {df_summary.shape[0]}")

=== Datasets Loaded Successfully ===
Individual Data Rows: 578
Summary Data Rows:    783


In [26]:
import pandas as pd

# 1. Ensure the date column is in datetime format
df['Date_Confirmation'] = pd.to_datetime(df['Date_Confirmation'])

# 2. Filter out 'Imported' cases to isolate local municipality transmission patterns
df_local = df[df['County_Living'] != 'Imported'].copy()

# 3. Pivot the data to create a daily grid: Dates as index, Counties as columns
df_daily_grid = df_local.pivot_table(
    index='Date_Confirmation',
    columns='County_Living',
    values='Number_of_Confirmed_Cases',
    aggfunc='sum'
).fillna(0)

# 4. Resample the daily data into weekly totals (resampling by 'W' aggregates by week ending Sunday)
df_weekly = df_daily_grid.resample('W').sum()

# 5. Let's look at the top 5 most affected municipalities to keep our baseline model highly focused
top_municipalities = df_weekly.sum().nlargest(5).index.tolist()
df_forecasting_data = df_weekly[top_municipalities]

print("=== Week-Over-Week Preprocessing Complete ===")
print(f"Time-series shape (Weeks x Municipalities): {df_forecasting_data.shape}")
print("\nTop Municipalities targeted for forecasting:")
print(top_municipalities)
print("\nLatest weekly case baselines:")
print(df_forecasting_data.head())

=== Week-Over-Week Preprocessing Complete ===
Time-series shape (Weeks x Municipalities): (145, 5)

Top Municipalities targeted for forecasting:
['New Taipei City', 'Taichung City', 'Taoyuan City', 'Kaohsiung City', 'Taipei City']

Latest weekly case baselines:
County_Living      New Taipei City  Taichung City  Taoyuan City  \
Date_Confirmation                                                 
2022-10-09                 65878.0        40800.0       34471.0   
2022-10-16                 60260.0        40337.0       32014.0   
2022-10-23                 49634.0        35020.0       26568.0   
2022-10-30                 42203.0        32548.0       23548.0   
2022-11-06                  5653.0         4732.0        3017.0   

County_Living      Kaohsiung City  Taipei City  
Date_Confirmation                               
2022-10-09                33448.0      34139.0  
2022-10-16                35170.0      30094.0  
2022-10-23                32396.0      24625.0  
2022-10-30             